# 04 · Human in the loop with interrupts

After this notebook you can pause a graph in the middle of a run with `interrupt()`, show a person exactly what is about to happen, and resume with approve, edit or reject, including for a risky tool call that a model proposed ("refund $500").

**Time:** ~30 min · **Runs:** offline on CPU in < 40 s (about 12 short LLM calls) · **Needs:** [03 · Checkpointing, memory and time travel](03_checkpointing_memory_and_time_travel.ipynb)

## Why this matters in interviews

- "Where does human-in-the-loop belong?" Before anything **irreversible**: moving money, deleting data, sending external messages, changing permissions. Everything else is better reviewed after the fact by sampling.
- HITL is a **durability** problem as much as a UX one. The approval may take hours and arrive at a different server, so the run must be checkpointed and resumable. Interviewers check that you know this.
- "Build a support agent that can issue refunds" and "an agent refunded money it shouldn't have" are classic design prompts. The expected answer: the model *proposes*, code and people *dispose*, with a durable approval gate, policy limits in code, idempotency keys and an audit trail. This notebook builds each piece.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag14` | What is human in the loop, and where does it belong? |
| `ag25` | How do you handle irreversible actions? |
| `ag36` | How would you build a customer support agent that can issue refunds? |
| `tl30` | An agent occasionally issues a refund it should not, and nobody can reconstruct why. What do you do? |
| `po28` | Design the path between an agent proposing a tool call and the call executing. |

## 1. Setup

Part A (sections 2–6) is plain Python so the mechanics are easy to see. Part B (sections 7–9) puts a model in front: it proposes the refund as a tool call and a human decides. Offline, the stand-in model reads the order id and amount out of the customer's message, which is all Part B needs.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import operator
from collections import Counter
from typing import Annotated, Literal, TypedDict

import matplotlib.pyplot as plt
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from llm_setup import langchain_chat_model

llm = langchain_chat_model(temperature=0)
print("chat model:", MODEL, "| provider:", PROVIDER)

The graph-drawing helper used throughout this section.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_graph(app, title=None, xray=False):
    """Draw a compiled LangGraph left to right: solid = fixed edge, dashed = conditional edge,
    arcs above = skip ahead, arcs below = loop back. Red text = the path-map label."""
    g = app.get_graph(xray=xray)
    edges = [(e.source, e.target, e.conditional, e.data) for e in g.edges]
    col, frontier = {"__start__": 0}, ["__start__"]
    while frontier:                                     # column = hops from START (BFS)
        nxt = []
        for n in frontier:
            for s, t, _, _ in edges:
                if s == n and t not in col and t != "__end__":
                    col[t] = col[n] + 1; nxt.append(t)
        frontier = nxt
    for n in g.nodes:
        col.setdefault(n, max(col.values()) + 1)
    col["__end__"] = max(c for n, c in col.items() if n != "__end__") + 1
    rows = {}
    for n in g.nodes:
        rows.setdefault(col[n], []).append(n)
    pos = {n: (c * 2.4, (len(ns) - 1) / 2 - i) for c, ns in rows.items() for i, n in enumerate(ns)}
    nrow = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(figsize=(min(11, 2.0 * len(rows) + 0.6), min(4.5, 0.9 * nrow + 1.4)))
    boxes = {}
    for n, (x, y) in pos.items():
        parts = [{"__start__": "START", "__end__": "END"}.get(p, p) for p in n.split(":")]
        fc = {"START": "#d3f9d8", "END": "#ffe3e3"}.get(parts[-1], "#e7f0ff")
        boxes[n] = FancyBboxPatch((x - 0.8, y - 0.28), 1.6, 0.56, boxstyle="round,pad=0.05", fc=fc, ec="#3b5bdb")
        ax.add_patch(boxes[n])
        ax.text(x, y, "\n".join(parts), ha="center", va="center", fontsize=8.5 if len(parts) == 1 else 7.5)
    pairs = {(s, t) for s, t, _, _ in edges}
    for s, t, cond, data in edges:
        (x1, y1), (x2, y2) = pos[s], pos[t]
        span = col[t] - col[s]
        if (t, s) in pairs and abs(span) == 1:
            rad = 0.2                                   # two-way pair: split the arrows apart
        elif span == 1 or (span == 0 and s != t):
            rad = 0.0
        else:
            rad = -0.3 if span > 1 else -0.45           # skip ahead above, loop back below
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), patchA=boxes[s], patchB=boxes[t], arrowstyle="-|>",
                                     mutation_scale=12, lw=1.3, color="#495057", linestyle="--" if cond else "-",
                                     connectionstyle=f"arc3,rad={rad}"))
        if data is not None and str(data) != t:
            mx, my = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1), (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
            ax.text(mx, my + 0.1, str(data), ha="center", fontsize=7.5, color="#c92a2a")
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 1.0, max(xs) + 1.0); ax.set_ylim(min(ys) - 0.75, max(ys) + 0.85)
    ax.axis("off"); ax.set_title(title or "graph", fontsize=10)
    plt.tight_layout(); plt.show()

## 2. `interrupt()`: pause inside a node

**In plain English:** `interrupt(payload)` stops the run at that line, saves a checkpoint, and hands `payload` (the question for the human) back to whoever called the graph. Later, calling the graph with `Command(resume=answer)` on the same thread restarts that node, and this time `interrupt()` **returns** `answer`.

```
 invoke(input) ──► load_order ──► approval: interrupt({...}) ──► returns {"__interrupt__": [...]}   (run paused, checkpoint saved)
                                         ... minutes or hours later ...
 invoke(Command(resume={"action": "approve"})) ──► approval: interrupt() returns the decision ──► execute ──► END
```

A fake payment system first. It enforces **one refund per order** with an idempotency key, so a retried call can never pay twice.

In [ ]:
ORDERS = {"ORD1042": 620.00, "ORD1043": 540.00, "ORD1044": 510.00, "ORD2001": 45.00, "ORD3001": 95.00}
PAYMENTS = {}                                    # idempotency key -> what was paid

def pay_refund(order_id: str, amount: float) -> str:
    key = f"refund:{order_id}"                   # one refund per order, however often we are called
    if key in PAYMENTS:
        return f"duplicate ignored (already paid {PAYMENTS[key]})"
    PAYMENTS[key] = f"${amount:.2f} on {order_id}"
    return f"refunded {PAYMENTS[key]}"

In [ ]:
BODY_RUNS = Counter()

class RefundState(TypedDict):
    order_id: str
    amount: float
    order_total: float
    status: str
    audit: Annotated[list, operator.add]

def load_order(state: RefundState) -> dict:
    return {"order_total": ORDERS[state["order_id"]], "audit": [f"loaded {state['order_id']}"]}

def approval(state: RefundState) -> Command[Literal["execute", "notify_customer"]]:
    BODY_RUNS["approval"] += 1                   # counts how often this function body starts
    decision = interrupt({"question": "Approve this refund?", "order_id": state["order_id"],
                          "amount": state["amount"], "order_total": state["order_total"]})
    who = decision.get("by", "unknown")
    if decision["action"] == "approve":
        return Command(goto="execute", update={"audit": [f"{who}: approved ${state['amount']:.0f}"]})
    if decision["action"] == "edit":
        return Command(goto="execute", update={"amount": decision["amount"],
                       "audit": [f"{who}: edited ${state['amount']:.0f} -> ${decision['amount']:.0f}"]})
    return Command(goto="notify_customer",
                   update={"status": "rejected", "audit": [f"{who}: rejected ({decision['reason']})"]})

def execute(state: RefundState) -> dict:
    return {"status": pay_refund(state["order_id"], state["amount"]), "audit": ["payment system called"]}

def notify_customer(state: RefundState) -> dict:
    return {"audit": ["customer told the refund was declined"]}

In [ ]:
rb = StateGraph(RefundState)
rb.add_node("load_order", load_order)
rb.add_node("approval", approval)
rb.add_node("execute", execute)
rb.add_node("notify_customer", notify_customer)
rb.add_edge(START, "load_order")
rb.add_edge("load_order", "approval")
rb.add_edge("execute", END)
rb.add_edge("notify_customer", END)

saver = InMemorySaver()
refunds = rb.compile(checkpointer=saver)
draw_graph(refunds, "Refund approval: 'approval' pauses with interrupt() and resumes with a decision")

In [ ]:
cfg = {"configurable": {"thread_id": "refund-1"}}
out = refunds.invoke({"order_id": "ORD1042", "amount": 500.0, "audit": []}, cfg)

print("keys returned :", list(out))
print("question      :", out["__interrupt__"][0].value)
snap = refunds.get_state(cfg)
print("paused before :", snap.next)
print("pending       :", [i.value["question"] for i in snap.interrupts])
print("paid so far   :", PAYMENTS)
assert snap.next == ("approval",) and PAYMENTS == {}

The run is paused *inside* `approval`, the question is stored in the checkpoint, and nothing has been paid. Now a reviewer answers:

In [ ]:
done = refunds.invoke(Command(resume={"action": "approve", "by": "alice"}), cfg)
print("status:", done["status"])
print("audit :", done["audit"])
assert done["status"] == "refunded $500.00 on ORD1042" and refunds.get_state(cfg).next == ()

## 3. Approve, edit, reject: three endings

Same graph, three threads, three decisions. The decision is ordinary data, so it can carry a new amount (edit) or a reason (reject), and the node routes on it with `Command`.

In [ ]:
cases = {
    "refund-2": ("ORD1043", {"action": "edit", "amount": 200.0, "by": "alice"}),
    "refund-3": ("ORD1044", {"action": "reject", "reason": "change of mind is not covered", "by": "bob"}),
}
for thread, (order, decision) in cases.items():
    c = {"configurable": {"thread_id": thread}}
    refunds.invoke({"order_id": order, "amount": 500.0, "audit": []}, c)          # pauses at approval
    final = refunds.invoke(Command(resume=decision), c)
    print(f"{thread}: {decision['action']:6s} -> status={final['status']!r}")
    print(f"          audit={final['audit']}")
print("payments:", PAYMENTS)
assert PAYMENTS["refund:ORD1043"] == "$200.00 on ORD1043" and "refund:ORD1044" not in PAYMENTS

## 4. The gotcha: the node restarts from the top

Resuming does not continue from the `interrupt()` line. LangGraph **re-runs the whole node** from its first line; this time `interrupt()` returns the resume value instead of pausing. The counter shows it:

In [ ]:
print("approval body started", BODY_RUNS["approval"], "times for", 3, "decisions")
assert BODY_RUNS["approval"] == 2 * 3

Each decision cost two starts: one that paused, one that resumed. So:

- Code **before** `interrupt()` in the same node runs again on resume. Keep it free of side effects (no payments, no emails, no counters that matter), or make it idempotent.
- Put side effects in a **separate node after** the approval, like `execute` above.
- With several `interrupt()` calls in one node, resume values are matched **by order**. Don't put interrupts inside branches or loops whose order can change between the first run and the resume.
- The payload must be JSON-serialisable: it is stored in the checkpoint and shown to a person.

## 5. Why human-in-the-loop needs a checkpointer

Without a checkpointer the graph can still *reach* the interrupt, but nothing is saved, so there is nothing to resume:

In [ ]:
no_memory = rb.compile()                                  # same graph, no checkpointer
paused = no_memory.invoke({"order_id": "ORD1042", "amount": 500.0, "audit": []})
print("reached the interrupt:", "__interrupt__" in paused)
try:
    no_memory.invoke(Command(resume={"action": "approve"}))
except RuntimeError as e:
    print("resume failed:", e)

With one, the approval can arrive much later and be handled by a **different process**. Here a second graph object (think: another worker after a deploy) resumes a thread the first one started. With `PostgresSaver` instead of `InMemorySaver`, the two could be on different machines.

In [ ]:
worker_1 = rb.compile(checkpointer=saver)
worker_2 = rb.compile(checkpointer=saver)
c = {"configurable": {"thread_id": "refund-4"}}
worker_1.invoke({"order_id": "ORD3001", "amount": 90.0, "audit": []}, c)          # pauses, worker_1 moves on
resumed = worker_2.invoke(Command(resume={"action": "approve", "by": "carol"}), c)
print("worker_2 finished what worker_1 started:", resumed["status"])
assert resumed["status"] == "refunded $90.00 on ORD3001"

## 6. Streaming until the interrupt

A UI streams the run to show progress; the last event before the pause is `{"__interrupt__": (...)}`, whose payload is exactly what to render on the approval card. Resuming with `stream` continues the event stream.

In [ ]:
c = {"configurable": {"thread_id": "refund-5"}}
PAYMENTS.pop("refund:ORD1042", None)                       # reset so the demo can pay ORD1042 again
for event in refunds.stream({"order_id": "ORD1042", "amount": 300.0, "audit": []}, c, stream_mode="updates"):
    for node, data in event.items():
        print(f"{node:15s}", data[0].value if node == "__interrupt__" else data)
print("--- reviewer clicks Approve ---")
for event in refunds.stream(Command(resume={"action": "approve", "by": "alice"}), c, stream_mode="updates"):
    for node, data in event.items():
        print(f"{node:15s}", data)

## 7. The real thing: review a tool call that a model proposed

Now the model decides *what should happen* and the system decides *what does happen*. The pattern:

```
            ┌──────── tool call? no ───────────────────────────────► END (the model's reply)
 START ─► agent ── yes ─► human_review ── approve / edit ─► tools ─┐
            ▲                  │  auto-approve if amount <= $100 ──┘ │
            │                  └── reject: a ToolMessage with the reason
            └──────────────────────────────────────────────────────────┘
```

- **Policy in code:** refunds up to `AUTO_APPROVE_LIMIT` go straight through; anything larger needs a person. The model cannot talk its way past an `if` statement.
- **Edit** replaces the model's tool call before it runs. Because `add_messages` replaces a message with the same `id`, the edited call *becomes* the call in the history.
- **Reject** answers the tool call with a `ToolMessage` carrying the reason. The API requires every tool call to get a tool result, and the model uses the reason to explain to the customer.
- **Audit:** every decision is appended to an `audit` list in state, so it is checkpointed with everything else.

In [ ]:
@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund to the customer's original payment method. Irreversible."""
    return pay_refund(order_id, amount)

SYSTEM = SystemMessage("You are a customer-support agent. Use issue_refund when a customer asks for a refund. "
                       "If a refund is rejected, apologise, explain the reason and do not retry.")
AUTO_APPROVE_LIMIT = 100.0
agent_llm = llm.bind_tools([issue_refund])

class SupportState(MessagesState):
    audit: Annotated[list, operator.add]

def agent(state: SupportState) -> dict:
    return {"messages": [agent_llm.invoke([SYSTEM, *state["messages"]])]}

def route_after_agent(state: SupportState) -> Literal["human_review", "__end__"]:
    return "human_review" if state["messages"][-1].tool_calls else END

In [ ]:
def human_review(state: SupportState) -> Command[Literal["tools", "agent"]]:
    proposal = state["messages"][-1]
    calls = proposal.tool_calls
    if all(c["args"].get("amount", 0) <= AUTO_APPROVE_LIMIT for c in calls):
        return Command(goto="tools", update={"audit": [f"policy: auto-approved {[c['args'] for c in calls]}"]})

    decision = interrupt({"question": "Run this tool call?", "calls": [{"name": c["name"], "args": c["args"]}
                                                                        for c in calls]})
    who = decision.get("by", "unknown")
    if decision["action"] == "approve":
        return Command(goto="tools", update={"audit": [f"{who}: approved {calls[0]['args']}"]})
    if decision["action"] == "edit":                 # edit applies to the first call (one call per turn here)
        edited = {**calls[0], "args": {**calls[0]["args"], **decision["args"]}}
        new_msg = AIMessage(content=proposal.content, tool_calls=[edited, *calls[1:]], id=proposal.id)
        return Command(goto="tools", update={"messages": [new_msg],
                                             "audit": [f"{who}: edited {calls[0]['args']} -> {edited['args']}"]})
    rejections = [ToolMessage(content=f"Rejected by a human reviewer: {decision['reason']}. Do not retry.",
                              tool_call_id=c["id"], name=c["name"]) for c in calls]
    return Command(goto="agent", update={"messages": rejections, "audit": [f"{who}: rejected ({decision['reason']})"]})

In [ ]:
sb = StateGraph(SupportState)
sb.add_node("agent", agent)
sb.add_node("human_review", human_review)
sb.add_node("tools", ToolNode([issue_refund]))
sb.add_edge(START, "agent")
sb.add_conditional_edges("agent", route_after_agent, ["human_review", END])
sb.add_edge("tools", "agent")
support = sb.compile(checkpointer=InMemorySaver())
draw_graph(support, "Support agent: every tool call passes human_review before it can run")

Four customers write in. The small refund goes straight through; the other three stop and wait.

In [ ]:
PAYMENTS.clear()
TICKETS = {"t-40": "Order ORD2001 arrived late - please refund $40.",
           "t-500": "Order ORD1042 arrived broken - please refund $500.",
           "t-edit": "Order ORD1043 is missing parts - please refund $500.",
           "t-reject": "Order ORD1044 - I changed my mind, refund $500."}
thread = lambda t: {"configurable": {"thread_id": t}}

for t, text in TICKETS.items():
    out = support.invoke({"messages": [HumanMessage(text)], "audit": []}, thread(t))
    print(f"{t:9s} {'PAUSED for review' if '__interrupt__' in out else 'finished: ' + out['messages'][-1].content}")
assert "refund:ORD2001" in PAYMENTS and len(PAYMENTS) == 1

### The reviewer inbox

Pending approvals live in the checkpoints, so an inbox is just "which threads have interrupts?". In production you would store thread ids in your own table and query `get_state` for each.

In [ ]:
inbox = []
for t in TICKETS:
    snap = support.get_state(thread(t))
    for pending in snap.interrupts:
        call = pending.value["calls"][0]
        inbox.append((t, call["name"], call["args"]["order_id"], call["args"]["amount"]))
print(f"{'thread':10s}{'tool':14s}{'order':9s}amount")
for row in inbox:
    print(f"{row[0]:10s}{row[1]:14s}{row[2]:9s}${row[3]:.2f}")
assert [r[0] for r in inbox] == ["t-500", "t-edit", "t-reject"]

In [ ]:
DECISIONS = {"t-500": {"action": "approve", "by": "alice"},
             "t-edit": {"action": "edit", "args": {"amount": 200.0}, "by": "alice"},
             "t-reject": {"action": "reject", "reason": "change of mind is outside the refund policy", "by": "bob"}}
for t, decision in DECISIONS.items():
    out = support.invoke(Command(resume=decision), thread(t))
    print(f"{t:9s} -> {out['messages'][-1].content}")

print("\npayments:", PAYMENTS)
assert PAYMENTS == {"refund:ORD2001": "$40.00 on ORD2001", "refund:ORD1042": "$500.00 on ORD1042",
                    "refund:ORD1043": "$200.00 on ORD1043"}

Offline, the stand-in's final replies simply quote the tool results. A real model turns the same tool results into a customer-facing message, and on the rejected thread it apologises and gives the reason, because the rejection arrived as the tool's answer.

### The audit trail and the idempotency key

The audit log answers "who decided what?" for every thread, and it lives in the checkpoint, next to the conversation it explains. That is the missing piece in "a refund went out and nobody can reconstruct why".

In [ ]:
for t in TICKETS:
    print(f"{t:9s}", support.get_state(thread(t)).values["audit"])

retry = issue_refund.invoke({"order_id": "ORD1042", "amount": 500.0})   # e.g. a client retry after a timeout
print("\nretried refund call:", retry)
assert retry.startswith("duplicate ignored") and len(PAYMENTS) == 3

## 8. Static breakpoints: `interrupt_before` and `interrupt_after`

You can also pause **at compile time**, before or after named nodes, without touching the node code. The run stops before the node, you inspect (or edit with `update_state`), then continue with `invoke(None, config)`. This is mainly a **debugging** tool: it pauses on every visit, carries no question payload, and takes no resume value.

In [ ]:
plain = StateGraph(MessagesState)
plain.add_node("agent", lambda s: {"messages": [agent_llm.invoke([SYSTEM, *s["messages"]])]})
plain.add_node("tools", ToolNode([issue_refund]))
plain.add_edge(START, "agent"); plain.add_conditional_edges("agent", tools_condition); plain.add_edge("tools", "agent")
debugger = plain.compile(checkpointer=InMemorySaver(), interrupt_before=["tools"])

c = thread("debug-1")
debugger.invoke({"messages": [HumanMessage("Order ORD3001 arrived broken - please refund $80.")]}, c)
snap = debugger.get_state(c)
print("stopped before:", snap.next, "| interrupts:", snap.interrupts)
print("pending call  :", snap.values["messages"][-1].tool_calls[0]["args"])
out = debugger.invoke(None, c)                             # continue
print("continued     :", out["messages"][-1].content)
assert snap.next == ("tools",) and snap.interrupts == ()

| | `interrupt()` in a node | `interrupt_before` / `interrupt_after` |
|---|---|---|
| Where you declare it | inside the node's code, at the exact line | at `compile()` (or per call), by node name |
| When it pauses | only when your code reaches it, e.g. only above $100 | every time the node is reached |
| What the human sees | your payload: the question, the proposed call | nothing: you inspect the state yourself |
| How you resume | `Command(resume=value)`; the value flows back into the node | `invoke(None, config)`, optionally after `update_state` |
| Best for | production approval, edit and clarification flows | debugging and stepping through a graph |

## 9. The prebuilt version: `HumanInTheLoopMiddleware`

If you build agents with LangChain's `create_agent`, the same gate is one line of middleware. It interrupts before the listed tools and resumes with a list of `decisions` (`approve`, `edit`, `reject`, or `respond`). Under the hood it is the same `interrupt()` and checkpointer you just used.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

PAYMENTS.clear()
prebuilt = create_agent(llm, [issue_refund], system_prompt=SYSTEM.content, checkpointer=InMemorySaver(),
                        middleware=[HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True})])
c = thread("prebuilt-1")
out = prebuilt.invoke({"messages": [HumanMessage("Order ORD1042 arrived broken - please refund $500.")]}, c)
request = out["__interrupt__"][0].value["action_requests"][0]
print("asks to approve:", request["name"], request["args"])
out = prebuilt.invoke(Command(resume={"decisions": [{"type": "edit", "edited_action": {
    "name": "issue_refund", "args": {"order_id": "ORD1042", "amount": 250.0}}}]}), c)
print("payments:", PAYMENTS)
assert PAYMENTS == {"refund:ORD1042": "$250.00 on ORD1042"}

## Try it yourself

**1.** An escalation ladder: add an `"escalate"` action to the Part A graph. It routes to a `manager_review` node that interrupts **again** with the reviewer's note; the manager can approve or reject. Resume twice and check both decisions are in the audit.

In [ ]:
# Solution — try it yourself first, then run this
def approval_with_escalation(state: RefundState) -> Command[Literal["execute", "notify_customer", "manager_review"]]:
    decision = interrupt({"question": "Approve this refund?", "order_id": state["order_id"], "amount": state["amount"]})
    if decision["action"] == "escalate":
        return Command(goto="manager_review", update={"audit": [f"{decision['by']}: escalated ({decision['note']})"]})
    return Command(goto="execute" if decision["action"] == "approve" else "notify_customer",
                   update={"audit": [f"{decision['by']}: {decision['action']}"]})

def manager_review(state: RefundState) -> Command[Literal["execute", "notify_customer"]]:
    decision = interrupt({"question": "Escalated refund: approve?", "amount": state["amount"]})
    return Command(goto="execute" if decision["action"] == "approve" else "notify_customer",
                   update={"audit": [f"{decision['by']} (manager): {decision['action']}"]})

eb = StateGraph(RefundState)
for name, fn in [("load_order", load_order), ("approval", approval_with_escalation),
                 ("manager_review", manager_review), ("execute", execute), ("notify_customer", notify_customer)]:
    eb.add_node(name, fn)
eb.add_edge(START, "load_order"); eb.add_edge("load_order", "approval")
eb.add_edge("execute", END); eb.add_edge("notify_customer", END)
ladder = eb.compile(checkpointer=InMemorySaver())
c = thread("ladder-1")
ladder.invoke({"order_id": "ORD1044", "amount": 480.0, "audit": []}, c)
ladder.invoke(Command(resume={"action": "escalate", "by": "alice", "note": "large amount"}), c)
final = ladder.invoke(Command(resume={"action": "approve", "by": "dana"}), c)
print(final["audit"]); print(final["status"])
assert final["status"] == "refunded $480.00 on ORD1044" and len(final["audit"]) == 4

**2.** Policy in code, before any human: add a `check_policy` node between `load_order` and `approval` that rejects a refund larger than the order total **without** interrupting. Show that a $5,000 refund on `ORD2001` ($45) never reaches a reviewer.

In [ ]:
# Solution — try it yourself first, then run this
def check_policy(state: RefundState) -> Command[Literal["approval", "__end__"]]:
    if state["amount"] > state["order_total"]:
        return Command(goto=END, update={"status": "auto-rejected: amount exceeds the order total",
                                         "audit": ["policy: rejected, no human needed"]})
    return Command(goto="approval")

pb = StateGraph(RefundState)
for name, fn in [("load_order", load_order), ("check_policy", check_policy), ("approval", approval),
                 ("execute", execute), ("notify_customer", notify_customer)]:
    pb.add_node(name, fn)
pb.add_edge(START, "load_order"); pb.add_edge("load_order", "check_policy")
pb.add_edge("execute", END); pb.add_edge("notify_customer", END)
guarded = pb.compile(checkpointer=InMemorySaver())
out = guarded.invoke({"order_id": "ORD2001", "amount": 5000.0, "audit": []}, thread("policy-1"))
print(out["status"], "| interrupted:", "__interrupt__" in out)
assert "__interrupt__" not in out and out["status"].startswith("auto-rejected")

**3.** Approvals must not wait forever. Write `expire_pending(app, thread_ids, reason)` that resumes every thread still waiting at an interrupt with a `reject` decision, and use it on two fresh Part A threads.

In [ ]:
# Solution — try it yourself first, then run this
def expire_pending(app, thread_ids, reason="approval timed out"):
    expired = []
    for t in thread_ids:
        if app.get_state(thread(t)).interrupts:
            app.invoke(Command(resume={"action": "reject", "reason": reason, "by": "system"}), thread(t))
            expired.append(t)
    return expired

for t, order in [("stale-1", "ORD1042"), ("stale-2", "ORD1043")]:
    refunds.invoke({"order_id": order, "amount": 100.0, "audit": []}, thread(t))
print("expired:", expire_pending(refunds, ["stale-1", "stale-2", "refund-1"]))
assert all(refunds.get_state(thread(t)).values["status"] == "rejected" for t in ["stale-1", "stale-2"])

## Say it in an interview

**30-second answer (where HITL belongs):** "Before irreversible actions: money, deletions, external messages, permission changes. I classify every tool as read, reversible write or irreversible, and only gate the last kind, because gating everything trains reviewers to click approve without reading. Mechanically it's a durable interrupt: the graph checkpoints, the approval request goes to an inbox, and the run resumes from the checkpoint when the decision arrives, possibly hours later on another server."

**30-second answer (the refund agent):** "The model decides what *should* happen; code decides what *does* happen. The model proposes a structured refund call. Deterministic policy runs first (amount within the order total, auto-approve below a threshold), anything above goes to a human who can approve, edit or reject, and a rejection goes back to the model as the tool result so it can explain. The refund call carries an idempotency key so a retry can't pay twice, and every proposal, decision and execution is in an audit trail next to the conversation."

**Facts to have ready:**
- `interrupt(payload)` pauses and returns `{"__interrupt__": [Interrupt(value=payload)]}`; resume with `invoke(Command(resume=value), config)`. Pending ones are visible in `get_state(config).interrupts` (and `.tasks`).
- Resuming **re-runs the node from the top**; side effects belong after the interrupt, in their own node, and must be idempotent.
- A checkpointer is mandatory to resume (`RuntimeError: Cannot use Command(resume=...) without checkpointer`).
- `interrupt_before` / `interrupt_after` are static breakpoints for debugging; continue with `invoke(None, config)`.
- `create_agent(..., middleware=[HumanInTheLoopMiddleware(interrupt_on={...})])` packages the same pattern.

**Follow-ups they ask:** *"What if nobody answers?"* Approvals expire: a job rejects (or escalates) stale interrupts, and the customer is told. *"How do you stop reviewers rubber-stamping?"* Gate only irreversible actions, show the exact arguments in plain terms, sample-audit the approvals. *"Why not just ask the model to confirm with the user?"* A prompt is probabilistic; an interrupt plus a policy check in code is deterministic.

**Traps:** a side effect before `interrupt()` in the same node (it runs twice), no idempotency key on the payment, letting a rejected tool call go unanswered (the next model call fails validation), and putting the approval limit in the prompt instead of in code.

## Next

- [05 · ReAct tool-calling agent](05_react_tool_calling_agent.ipynb): the agent loop this approval gate sits inside.
- [Guardrails, HITL and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb) · [guardrails for PII and prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb) · [build an agent in 30 minutes](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb)
- Theory: [LangGraph course](../../langgraph/index.html) (chapter 9, interrupts and HITL), [agentic AI flow](../../agentic_ai_flow/index.html), [interview bank](../../ai_interview_prep/index.html).